# Snowflake Anomaly Monitors（Preview）— 検証ノートブック

## このノートブックについて

Zenn 記事「Snowflakeのコスト異常検知をスコープ別に設定できるようになった——Anomaly Monitors（Preview）を実機検証」
のハンズオンで実行した SQL をまとめたものです。

Snowflake Notebooks にインポートすると、そのまま自分の環境で実行できます。

### 注意

- `ACCOUNTADMIN` 相当の権限が必要です
- monitor はアカウントあたり 20 個までです。既存の monitor がある場合は上限に注意してください
- 最後にクリーンアップのセルがあります。**必ず実行して作成した monitor を削除してください**
- 記事の「ステップ5」は Snowsight の操作なので、このノートブックには含まれません


## セットアップ

ロールを設定します。


In [ ]:
USE ROLE ACCOUNTADMIN;


現在の monitor 一覧を確認します。

空でない場合、20 個の上限に注意してください。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!LIST_MONITORS();


## ステップ1: monitor を作成する

`CREATE_MONITOR` の引数は「別名」と「設定オブジェクト」の2つです。

設定はすべて第2引数の `OBJECT` にまとめます。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!CREATE_MONITOR(
    'ML_WH_MONITOR',
    OBJECT_CONSTRUCT(
        'credit_family', 'CREDITS',
        'service_types', ARRAY_CONSTRUCT('WAREHOUSE_METERING')
    )
);


戻り値は正規化された設定オブジェクトです。

指定しなかった `resource_tags` に既定値が埋まります。


## ステップ2: 判定結果を確認する

作成直後から、過去に遡って判定結果が返ります。

`UPPER_BOUND` が日ごとに変動していることを確認してください。閾値は一切設定していません。

日付は自分の環境に合わせて変更してください。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!GET_MONITOR_ANOMALIES(
    'ML_WH_MONITOR', '2026-08-01', '2026-08-15'
);


## ステップ3: 通知先を設定する

monitor ごとに独立した通知先を設定します。

第2引数は配列ではなく**文字列**です。配列を渡すと `Invalid argument types` になります。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!SET_MONITOR_NOTIFICATION_EMAILS(
    'ML_WH_MONITOR', 'ml-team@example.com'
);


設定内容を確認します。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!GET_MONITOR_NOTIFICATION_EMAILS('ML_WH_MONITOR');


アカウント共通の通知先は別で管理されています。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!GET_ACCOUNT_NOTIFICATION_EMAILS();


## ステップ4: AI クレジット専用の monitor を作る

各 monitor は `CREDITS` か `AI_CREDITS` のどちらか一方だけを追跡します。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!CREATE_MONITOR(
    'AI_MONITOR',
    OBJECT_CONSTRUCT(
        'credit_family', 'AI_CREDITS',
        'service_types', ARRAY_CONSTRUCT('CORTEX_SEARCH', 'CORTEX_AGENTS', 'AI_FUNCTIONS')
    )
);


## 注意ポイント1: 通常クレジットと AI クレジットは合算できない

**以下の2つのセルは意図的にエラーになります。**

まず `credit_family` に両方を配列で渡すパターンです。`INVALID_MONITOR_CONFIG` になります。


In [ ]:
-- 意図的にエラーになるセル
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!CREATE_MONITOR(
    'BOTH_MONITOR',
    OBJECT_CONSTRUCT(
        'credit_family', ARRAY_CONSTRUCT('CREDITS', 'AI_CREDITS'),
        'service_types', ARRAY_CONSTRUCT('WAREHOUSE_METERING')
    )
);


次に、片方の `credit_family` に両家系のサービスタイプを混ぜるパターンです。

こちらも `INVALID_MONITOR_CONFIG` になります。
`credit_family` と `service_types` は同じ家系で揃っている必要があります。


In [ ]:
-- 意図的にエラーになるセル
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!CREATE_MONITOR(
    'MIX_MONITOR',
    OBJECT_CONSTRUCT(
        'credit_family', 'CREDITS',
        'service_types', ARRAY_CONSTRUCT('WAREHOUSE_METERING', 'CORTEX_SEARCH')
    )
);


## 注意ポイント2: タグスコープは SQL からは作れない

**以下の2つのセルも意図的にエラーになります。**

Snowsight で作った monitor が保存している形式をそのまま渡すと、
`Each tag pair must be an array: [tagReference, tagValue]` で拒否されます。

タグ名は自分の環境のものに置き換えてください。


In [ ]:
-- 意図的にエラーになるセル
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!CREATE_MONITOR(
    'SQL_TAG_TEST',
    OBJECT_CONSTRUCT(
        'credit_family', 'CREDITS',
        'service_types', ARRAY_CONSTRUCT(),
        'resource_tags', OBJECT_CONSTRUCT(
            'operator', 'UNION',
            'tags', ARRAY_CONSTRUCT(
                OBJECT_CONSTRUCT('tagDatabase','YOUR_DB', 'tagSchema','TAGS',
                                 'tagName','COST_CENTER', 'tagValues',ARRAY_CONSTRUCT('analytics'))
            )
        )
    )
);


エラーが要求する `[tagReference, tagValue]` の形で渡すと、今度はタグの解決で失敗します。

`Object '...' does not exist or not authorized` になります。

実在するタグを指定しても、存在しないタグを指定しても、同じエラーになります。


In [ ]:
-- 意図的にエラーになるセル
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!CREATE_MONITOR(
    'SQL_TAG_TEST2',
    OBJECT_CONSTRUCT(
        'credit_family', 'CREDITS',
        'service_types', ARRAY_CONSTRUCT('WAREHOUSE_METERING'),
        'resource_tags', OBJECT_CONSTRUCT(
            'operator', 'UNION',
            'tags', ARRAY_CONSTRUCT(
                ARRAY_CONSTRUCT('YOUR_DB.TAGS.COST_CENTER', 'analytics')
            )
        )
    )
);


## 補足: service_types に指定できる値を確かめる

指定できない値を渡すと `INVALID_MONITOR_CONFIG` になります。

この挙動を使って、自分の環境で使える値を確かめられます。
作成に成功した monitor はその場で削除しています。


In [ ]:
EXECUTE IMMEDIATE $$
DECLARE
  result STRING DEFAULT '';
  cands ARRAY DEFAULT ARRAY_CONSTRUCT(
    'WAREHOUSE_METERING','AUTO_CLUSTERING','MATERIALIZED_VIEW','SEARCH_OPTIMIZATION',
    'PIPE','SNOWPIPE_STREAMING','REPLICATION','SERVERLESS_TASK','SERVERLESS_ALERTS',
    'QUERY_ACCELERATION','COPY_FILES','LOGGING','TELEMETRY_DATA_INGEST',
    'HYBRID_TABLE_REQUESTS','DATA_QUALITY_MONITORING','TRUST_CENTER',
    'SNOWPARK_CONTAINER_SERVICES','AI_SERVICES',
    'AI_FUNCTIONS','CORTEX_SEARCH','CORTEX_AGENTS','SNOWFLAKE_COCO',
    'SNOWFLAKE_COCO_CLI','SNOWFLAKE_COCO_SNOWSIGHT','SNOWFLAKE_COWORK'
  );
  fams ARRAY DEFAULT ARRAY_CONSTRUCT('CREDITS','AI_CREDITS');
BEGIN
  FOR fi IN 0 TO ARRAY_SIZE(fams) - 1 DO
    LET fam STRING := GET(fams, fi)::STRING;
    result := result || '## ' || fam || ': ';
    FOR ci IN 0 TO ARRAY_SIZE(cands) - 1 DO
      LET st STRING := GET(cands, ci)::STRING;
      BEGIN
        CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!CREATE_MONITOR(
          'PROBE_TMP',
          OBJECT_CONSTRUCT('credit_family', :fam, 'service_types', ARRAY_CONSTRUCT(:st))
        );
        result := result || st || ',';
        CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!DROP_MONITOR('PROBE_TMP');
      EXCEPTION
        WHEN OTHER THEN
          result := result || '';
      END;
    END FOR;
    result := result || ' | ';
  END FOR;
  RETURN result;
END;
$$;


## クリーンアップ

**必ず実行してください。**

作成した monitor を削除します。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!DROP_MONITOR('ML_WH_MONITOR');


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!DROP_MONITOR('AI_MONITOR');


削除されたことを確認します。


In [ ]:
CALL SNOWFLAKE.LOCAL.ANOMALY_INSIGHTS!LIST_MONITORS();
